In [22]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

In [2]:
transformer = AutoModel.from_pretrained("google-bert/bert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
topk = 3

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
text = "The Eiffel Tower is located in Paris."

tokens = tokenizer(text)

In [4]:
tokens

{'input_ids': [101, 1996, 1041, 13355, 2884, 3578, 2003, 2284, 1999, 3000, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [5]:
type(tokens)

transformers.tokenization_utils_base.BatchEncoding

In [6]:
tokenizer.convert_ids_to_tokens(tokens["input_ids"])

['[CLS]',
 'the',
 'e',
 '##iff',
 '##el',
 'tower',
 'is',
 'located',
 'in',
 'paris',
 '.',
 '[SEP]']

In [7]:
len(tokenizer.convert_ids_to_tokens(tokens["input_ids"]))

12

In [8]:
inputs = tokenizer(text, return_tensors="pt")

outputs = transformer(**inputs)

In [9]:
type(outputs)

transformers.modeling_outputs.BaseModelOutputWithPoolingAndCrossAttentions

In [10]:
outputs

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.0617, -0.1000,  0.0535,  ..., -0.4519,  0.4607,  0.4501],
         [-0.1182,  0.2657, -0.2889,  ..., -0.0138,  0.7281,  0.0668],
         [ 0.4513, -0.2866,  0.6781,  ..., -0.0717,  0.6819, -0.3085],
         ...,
         [ 0.2230, -0.1653,  0.2300,  ..., -0.6433, -0.6355, -0.0829],
         [ 0.7864, -0.0589, -0.2918,  ...,  0.0365, -0.3541, -0.2074],
         [ 0.1276, -0.1130, -0.0310,  ...,  0.1753, -0.3551, -0.5028]]],
       grad_fn=<NativeLayerNormBackward0>), pooler_output=tensor([[-0.9631, -0.5305, -0.6778,  0.8217,  0.4774, -0.2807,  0.8381,  0.4724,
         -0.4639, -1.0000, -0.4530,  0.9016,  0.9946, -0.0772,  0.9326, -0.6818,
         -0.4002, -0.5851,  0.4594, -0.1632,  0.6998,  1.0000,  0.0653,  0.4651,
          0.5477,  0.9706, -0.8281,  0.9563,  0.9724,  0.8429, -0.7103,  0.4740,
         -0.9970, -0.2270, -0.6978, -0.9960,  0.6103, -0.7824,  0.0725, -0.2942,
         -0.9223,  0.5649,  1.00

In [11]:
outputs.last_hidden_state

tensor([[[-0.0617, -0.1000,  0.0535,  ..., -0.4519,  0.4607,  0.4501],
         [-0.1182,  0.2657, -0.2889,  ..., -0.0138,  0.7281,  0.0668],
         [ 0.4513, -0.2866,  0.6781,  ..., -0.0717,  0.6819, -0.3085],
         ...,
         [ 0.2230, -0.1653,  0.2300,  ..., -0.6433, -0.6355, -0.0829],
         [ 0.7864, -0.0589, -0.2918,  ...,  0.0365, -0.3541, -0.2074],
         [ 0.1276, -0.1130, -0.0310,  ...,  0.1753, -0.3551, -0.5028]]],
       grad_fn=<NativeLayerNormBackward0>)

In [12]:
outputs.last_hidden_state.shape

torch.Size([1, 12, 768])

In [13]:
outputs.last_hidden_state[0].shape

torch.Size([12, 768])

In [14]:
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

In [15]:
tokens

['[CLS]',
 'the',
 'e',
 '##iff',
 '##el',
 'tower',
 'is',
 'located',
 'in',
 'paris',
 '.',
 '[SEP]']

In [16]:
inputs["input_ids"][0]

tensor([  101,  1996,  1041, 13355,  2884,  3578,  2003,  2284,  1999,  3000,
         1012,   102])

In [17]:
for token, embedding in zip(tokens, outputs.last_hidden_state[0]):
    print(token, embedding.shape)

[CLS] torch.Size([768])
the torch.Size([768])
e torch.Size([768])
##iff torch.Size([768])
##el torch.Size([768])
tower torch.Size([768])
is torch.Size([768])
located torch.Size([768])
in torch.Size([768])
paris torch.Size([768])
. torch.Size([768])
[SEP] torch.Size([768])


In [45]:
def encode(texts: list[str]):

    inputs = tokenizer(texts, return_tensors="pt", padding=True)

    with torch.no_grad():
        outputs = transformer(**inputs)

    return F.normalize(outputs.last_hidden_state.squeeze(0), p=2, dim=1)

In [46]:
text1 = ["The Eiffel Tower is located in Paris.", "How many Towers in Paris?"]
query = "How many Towers in Paris?"

In [47]:
emb_doc = encode(text1)

In [48]:
emb_doc.shape

torch.Size([2, 12, 768])

In [26]:
def similarity_matrix(emb_doc, emb_query):

    similarity = emb_query @ emb_doc.T

    return similarity

In [81]:
emb_query = encode(query)

In [28]:
emb_doc.shape

torch.Size([12, 768])

In [29]:
emb_query.shape

torch.Size([8, 768])

In [31]:
similarity = similarity_matrix(emb_doc=emb_doc, emb_query=emb_query)

In [32]:
similarity.shape

torch.Size([8, 12])

In [37]:
similarity

tensor([[ 0.8746,  0.3970,  0.2614, -0.0859,  0.0107,  0.3596,  0.1066,  0.0995,
          0.1237,  0.0742, -0.0809, -0.0752],
        [ 0.6012,  0.5334,  0.4005, -0.0632,  0.1291,  0.4020,  0.1646,  0.1727,
          0.1604,  0.0844, -0.0179,  0.0414],
        [ 0.1342,  0.4440,  0.2621,  0.2599,  0.4533,  0.4242,  0.4982,  0.4914,
          0.4296,  0.3524,  0.1724,  0.2465],
        [ 0.2050,  0.4227,  0.2885,  0.2219,  0.3633,  0.6144,  0.4772,  0.4962,
          0.4364,  0.3410,  0.0633,  0.0561],
        [ 0.1727,  0.4802,  0.2835,  0.2340,  0.4423,  0.4378,  0.5412,  0.5544,
          0.6837,  0.3721,  0.1612,  0.2832],
        [ 0.1152,  0.3174,  0.3643,  0.3181,  0.5119,  0.3491,  0.4159,  0.4272,
          0.4810,  0.8133,  0.1005,  0.1725],
        [ 0.1367,  0.2848,  0.1278,  0.0704,  0.2120,  0.2371,  0.3574,  0.3324,
          0.2901,  0.1988,  0.1824,  0.1327],
        [-0.1128,  0.0564, -0.0013,  0.1286,  0.1240,  0.0570,  0.0881,  0.0790,
          0.0852,  0.1087,  0.

In [ ]:
def max_sim(similarity_matrix):

    return similarity_matrix.max(dim=1).values.sum()

In [38]:
sum = 0
for i in similarity:
    print(max(i))
    sum += max(i)

tensor(0.8746)
tensor(0.6012)
tensor(0.4982)
tensor(0.6144)
tensor(0.6837)
tensor(0.8133)
tensor(0.3574)
tensor(0.9012)


In [35]:
sum

tensor(5.3439)

In [69]:
chunks = [{"text": "aasdfasfdasfd", "index": "1"}, {"text": "5643ghfdfd", "index": "2"}]

In [71]:
data = []

for doc in chunks:
    data.append(doc["text"])

In [72]:
data

['aasdfasfdasfd', '5643ghfdfd']

In [73]:
embedding = encode(data)

In [75]:
embedding.shape

torch.Size([2, 9, 768])

In [ ]:
for emb in embedding:
    print(emb.shape)

torch.Size([9, 768])
torch.Size([9, 768])


In [79]:
tensors = []

for chunk, emb in zip(chunks, embedding):
    dict_data = {}
    dict_data["index"] = chunk["index"]
    dict_data["embedding"] = emb
    tensors.append(dict_data)

In [80]:
tensors

[{'index': '1',
  'embedding': tensor([[-0.4038,  0.0784,  0.1701,  ...,  0.1021,  0.1820,  0.5904],
          [-0.0109,  0.8499,  0.1925,  ...,  0.0622,  0.2480,  0.5378],
          [-0.1734,  0.3845,  0.5620,  ..., -0.4595, -0.3856,  0.0442],
          ...,
          [ 0.0453,  0.0177,  0.4351,  ...,  0.3820, -0.3652, -0.2287],
          [-0.3441,  0.1016, -0.0734,  ..., -0.1764, -0.1358,  0.3901],
          [ 0.8207,  0.0350, -0.0904,  ...,  0.5859, -0.6101, -0.3508]])},
 {'index': '2',
  'embedding': tensor([[-0.2774, -0.0987,  0.1370,  ..., -0.0993,  0.1528,  0.4334],
          [ 0.5674, -0.8754,  0.1469,  ..., -0.3579,  0.2738,  0.4003],
          [-0.0452, -0.1030,  0.3687,  ..., -0.6601, -0.2211, -0.0778],
          ...,
          [-0.1785, -0.0590,  0.0138,  ..., -0.0943, -0.1214,  0.1306],
          [ 0.6360,  0.2527, -0.1247,  ...,  0.2540, -0.4054, -0.2637],
          [-0.1723, -0.2587,  0.2040,  ..., -0.1443,  0.1598, -0.0278]])}]